# 02 — Feature Engineering

Exploratory analysis of synthetic behavioral dataset.
Goal: understand distributions, correlations, and confounds before training.

In [ ]:
import sys
from pathlib import Path

repo_root = Path(".").resolve().parent
sys.path.insert(0, str(repo_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.risk_classifier import FEATURE_COLS, DEMOGRAPHIC_COLS, TARGET_COL

df = pd.read_csv(repo_root / "data" / "synthetic" / "student_wellbeing.csv")
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
print(f"Label prevalence: {df[TARGET_COL].mean():.4f}")
df.head(3)

## 1. Feature Distributions by Label

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(FEATURE_COLS):
    ax = axes[i]
    for label, color, name in [(0, "steelblue", "No recommendation"), (1, "tomato", "Recommended")]:
        subset = df[df[TARGET_COL] == label][feat]
        ax.hist(subset, bins=40, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(feat, fontsize=10)
    ax.legend(fontsize=7)
    ax.set_xlabel("")

plt.suptitle("Feature distributions by support_recommended label", y=1.01)
plt.tight_layout()
plt.savefig(repo_root / "notebooks" / "feature_distributions.png", dpi=120, bbox_inches="tight")
plt.show()

## 2. Correlation Matrix (Features + Label)

In [ ]:
corr_cols = FEATURE_COLS + [TARGET_COL]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, ax=ax, square=True, linewidths=0.5)
ax.set_title("Feature correlation matrix")
plt.tight_layout()
plt.savefig(repo_root / "notebooks" / "feature_correlation.png", dpi=120, bbox_inches="tight")
plt.show()

# Features most correlated with label
label_corr = corr[TARGET_COL].drop(TARGET_COL).sort_values(key=abs, ascending=False)
print("Feature correlation with label:")
print(label_corr.round(3))

## 3. Demographic Confounds Analysis

Shows how demographic group membership correlates with behavioral features —
the realistic confounds built into the generator. Phase 5 bias audit will test
whether these confounds propagate into model predictions.

In [ ]:
# Positive rate by demographic group
for demo_col in ["gender", "race_ethnicity", "first_gen", "financial_aid_status"]:
    rates = df.groupby(demo_col)[TARGET_COL].mean().sort_values(ascending=False)
    print(f"\nPositive rate by {demo_col}:")
    for k, v in rates.items():
        print(f"  {k}: {v:.3f}")
    spread = rates.max() - rates.min()
    flag = " ⚠️  > 5pp spread" if spread > 0.05 else " ✓"
    print(f"  Spread: {spread:.3f}{flag}")

In [ ]:
# Financial stress confound
print("Financial stress rate by aid status:")
print(df.groupby("financial_aid_status")["financial_stress_flag"].mean().round(3))

print("\nEngagement variance: first-gen vs continuing-gen:")
print(df.groupby("first_gen")["engagement_variance"].describe().round(3))

print("\nMissed class streak: first-gen vs continuing-gen:")
print(df.groupby("first_gen")["missed_class_streak"].mean().round(3))

## 4. Class Imbalance Check

In [ ]:
label_counts = df[TARGET_COL].value_counts()
print(f"Class distribution:\n{label_counts}")
print(f"\nImbalance ratio: {label_counts[0] / label_counts[1]:.1f}:1")
print("SMOTE will be applied to training split to balance classes before fitting.")

## 5. Feature Importance Preview (Mutual Information)

In [ ]:
from sklearn.feature_selection import mutual_info_classif

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({"feature": FEATURE_COLS, "mutual_info": mi_scores})
mi_df = mi_df.sort_values("mutual_info", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(mi_df["feature"], mi_df["mutual_info"], color="steelblue")
ax.set_xlabel("Mutual information with label")
ax.set_title("Feature relevance (mutual information)")
plt.tight_layout()
plt.savefig(repo_root / "notebooks" / "feature_mutual_info.png", dpi=120, bbox_inches="tight")
plt.show()
print(mi_df.sort_values("mutual_info", ascending=False).to_string(index=False))